# N11 Tape Seed Ablation

Runs seed ablations for the tape model family using the `2d_tape_ICNN.ipynb` configuration. The default focuses on the anisotropic structured Brazier ICNN model; uncomment the alternate architecture list to broaden the sweep.

In [ ]:
import os
from pathlib import Path

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import TapeN11Properties
from run_architectures import SweepConfig

ROOT = Path.cwd()
train_file = "../experiment_data/tape_data/11_noded/n11_tape_train_dataset.npz"
valid_file = "../experiment_data/tape_data/11_noded/n11_tape_test_dataset.npz"
properties = TapeN11Properties(mass=-0.005)

# Copied from 2d_tape_ICNN.ipynb
K_init_chol = (0.2, 0.0, 0.1)
K_init_diag = (0.2, 0.1)

base_cfg = SweepConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10,),
    corr_factor=0.01,
    input_mode="raw",
    only_stretching_NN=False,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    mode="anisotropic",
    n_epochs=1000,
    lr=5e-2,
    weight_decay=1e-5,
    seed=42,
    valid_every=10,
    max_dlambda=5e-2,
    iters=10,
    ls_steps=10,
    abs_tol=5e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=True,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-6,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key=None,
    force_loss_strength=0.0,
    force_components=(0, 1, 2),
    force_sign=1.0,
    return_loss_components=False,
    early_stopping=True,
    early_stopping_patience=200,
    early_stopping_min_delta=1e-5,
    restore_best_model=True,
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_force_predictions=False,
    plot_force_predictions=False,
    save_hessian_diagnostics=False,
    save_energy_landscapes=False,
    verbose=True,
    continue_on_failure=True,
)

print(properties)

OUTPUT_DIR = ROOT / "seed_ablation_outputs_n11_tape"
SUMMARY_DIR = OUTPUT_DIR / "seed_ablation_summary"
base_cfg = base_cfg.__class__(**{
    **base_cfg.__dict__,
    "output_dir": str(OUTPUT_DIR),
    "seed_list": tuple(range(5)),
})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving seed ablation results under: {OUTPUT_DIR.resolve()}")


In [ ]:
from run_architectures import subset_brazier_stiffness_only, subset_tape_tube_candidates

selected_architectures = ["brazier_chol_stiffness_icnn"]

# Broader tape seed ablation options:
# selected_architectures = [
#     "brazier_chol_stiffness_baseline",
#     "brazier_chol_stiffness_mlp",
#     "brazier_chol_stiffness_icnn",
# ]
# selected_architectures = subset_tape_tube_candidates()

print(f"Running {len(selected_architectures)} architectures across seeds {base_cfg.seed_list}:")
for name in selected_architectures:
    print(f"  - {name}")


In [ ]:
from seed_ablation_utils import run_seed_ablation

all_seed_results = run_seed_ablation(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    base_cfg=base_cfg,
    selected_architectures=selected_architectures,
)


In [ ]:
from seed_ablation_utils import make_seed_ablation_plots

make_seed_ablation_plots(
    all_seed_results,
    output_dir=str(SUMMARY_DIR),
    traj_idx=0,
    x_idx=4,
    z_idx=6,
)
print("Seed ablation plots written to:", SUMMARY_DIR.resolve())


In [ ]:
from seed_ablation_utils import run_seed_hessian_diagnostics

run_seed_hessian_diagnostics(
    str(OUTPUT_DIR),
    use_predicted=True,
    splits=("train", "valid"),
    max_trajectories=1,
    stride=10,
    properties_class="TapeN11Properties",
)
print("Hessian diagnostics table:", OUTPUT_DIR / "hessian_diagnostics_table.csv")


In [ ]:
print("Done. Key outputs:")
print("  - <architecture>__seed_*/results.npz")
print("  - seed_ablation_summary/*.json and *.png")
print("  - hessian_diagnostics_table.csv")
